# 04 — Task-Based Model Selection — Practical Examples

**Covers**: Task classifier, escalation chain, multi-model routing, quality-gated selection, benchmarking by task type

In [ ]:
import os, json, time
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print('✅ Setup complete')

## 1. Task Classifier — Route to the Right Model

In [ ]:
class TaskProfile(BaseModel):
    task_type: Literal['extraction','coding','reasoning','summarization','classification','qa','chat','creative']
    complexity: Literal['simple','moderate','complex']
    context_length: Literal['short','medium','long']  # short<4k, medium<64k, long>64k
    real_time: bool
    privacy_required: bool
    recommended_model: str
    budget_model: str
    reason: str

MODEL_ROUTING = {
    ('extraction',     'simple',   False, False): ('gpt-4o-mini',      'gemini-1.5-flash'),
    ('extraction',     'complex',  False, False): ('gpt-4o',           'gpt-4o-mini'),
    ('coding',         'moderate', False, False): ('claude-3-5-sonnet','gpt-4o-mini'),
    ('coding',         'complex',  False, False): ('claude-3-5-sonnet','gpt-4o'),
    ('reasoning',      'complex',  False, False): ('gpt-4o',           'gpt-4o-mini'),
    ('summarization',  'simple',   False, False): ('gpt-4o-mini',      'gemini-1.5-flash'),
    ('classification', 'simple',   False, False): ('gpt-4o-mini',      'gemini-1.5-flash'),
    ('chat',           'moderate', True,  False): ('claude-3-5-haiku', 'gemini-2.0-flash'),
    ('qa',             'complex',  False, False): ('claude-3-5-sonnet','gpt-4o'),
}

def route_task(user_request: str) -> TaskProfile:
    r = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Classify this task and recommend an LLM model.'},
            {'role': 'user',   'content': user_request}
        ],
        response_format=TaskProfile
    )
    return r.choices[0].message.parsed

test_tasks = [
    'Extract all invoice fields (vendor, date, total, line items) from this PDF text.',
    'Write a Python function implementing a red-black tree with insert and search.',
    'What is photosynthesis? (chat message from user)',
    'Summarize this quarterly earnings report into 5 bullet points.',
    'Is this email spam or legitimate? Classify with confidence score.',
]

print(f"{'Task':<52} {'Type':<14} {'Complexity':<12} {'Model'}")
print('-' * 100)
for task in test_tasks:
    profile = route_task(task)
    print(f"{task[:50]:<52} {profile.task_type:<14} {profile.complexity:<12} {profile.recommended_model}")

## 2. Quality-Based Escalation Chain

In [ ]:
def evaluate_code_quality(code: str) -> float:
    """Simple code quality heuristic (0-1). Production: use LLM-as-judge."""
    score = 0.0
    if 'def ' in code:          score += 0.20
    if '"""' in code:           score += 0.20  # docstring
    if '->' in code:            score += 0.15  # return type
    if 'return' in code:        score += 0.15
    if len(code.split('\n'))>5: score += 0.15  # non-trivial
    if 'raise' in code or 'if not' in code: score += 0.15  # error handling
    return score

ECALATION_CHAIN = ['gpt-4o-mini', 'gpt-4o']
MIN_QUALITY = 0.70

TASK = 'Write a Python function binary_search(arr: list[int], target: int) -> int that returns the index or -1. Use O(log n) time. Include docstring, type hints, and edge case handling.'

print(f"Task: {TASK[:80]}...")
print(f"Min acceptable quality: {MIN_QUALITY}\n")

for model_id in ECDATION_CHAIN := ESCALATION_CHAIN:
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=model_id,
        messages=[{'role': 'system', 'content': 'Write complete, production-quality Python code.'},
                  {'role': 'user',   'content': TASK}],
        max_tokens=300, temperature=0
    )
    elapsed = (time.perf_counter() - t0) * 1000
    code = r.choices[0].message.content
    quality = evaluate_code_quality(code)
    cost = r.usage.total_tokens * 0.15e-6
    
    print(f"[{model_id}] quality={quality:.2f} latency={elapsed:.0f}ms cost=${cost:.5f}")
    if quality >= MIN_QUALITY:
        print(f"  ✅ Quality meets bar — using {model_id}")
        print(f"  Code: {code[:200]}")
        break
    else:
        print(f"  ⬆️  Quality below bar — escalating...")

## 3. Multi-Model Agent — Different Models for Different Subtasks

In [ ]:
# Architecture: cheap model for routing/summarization, quality model for reasoning
AGENT_CONFIG = {
    'router':    'gpt-4o-mini',   # Classify the request
    'extractor': 'gpt-4o-mini',   # Extract structured data
    'reasoner':  'gpt-4o',        # Reason and plan
    'coder':     'gpt-4o-mini',   # Generate code (flagship optional)
}

class AgentRequest(BaseModel):
    subtask: Literal['extract', 'plan', 'code', 'summarize']
    content: str

def multi_model_agent(user_query: str) -> dict:
    """Route to appropriate model based on classified subtask."""
    # Step 1: Classify (cheap model)
    route = client.beta.chat.completions.parse(
        model=AGENT_CONFIG['router'],
        messages=[{'role': 'system', 'content': 'Classify the user request into one of: extract, plan, code, summarize.'},
                  {'role': 'user',   'content': user_query}],
        response_format=AgentRequest
    )
    routed = route.choices[0].message.parsed
    
    # Step 2: Execute with appropriate model
    model_map = {'extract': AGENT_CONFIG['extractor'],
                 'plan':    AGENT_CONFIG['reasoner'],
                 'code':    AGENT_CONFIG['coder'],
                 'summarize': AGENT_CONFIG['extractor']}
    exec_model = model_map.get(routed.subtask, AGENT_CONFIG['router'])
    
    r = client.chat.completions.create(
        model=exec_model,
        messages=[{'role': 'user', 'content': user_query}],
        max_tokens=200
    )
    
    return {'routed_to': routed.subtask, 'model_used': exec_model,
            'result': r.choices[0].message.content[:150]}

tests = [
    'Extract the date and total from: Invoice #102, Jan 15 2025, Total $1,250.',
    'Write a function that sorts a list of dicts by a given key.',
    'Summarize: Python is popular for AI because of its readable syntax and rich ecosystem.',
]
for q in tests:
    res = multi_model_agent(q)
    print(f"Query: {q[:55]}")
    print(f"  → task={res['routed_to']}, model={res['model_used']}")
    print(f"  → {res['result'][:80]}")
    print()

## 📌 Summary

- **TaskProfile classifier**: categorize task → route to best model automatically
- **Escalation chain**: start cheap, validate quality, escalate if needed
- **Multi-model agents**: cheap model for routing/extraction, quality model for reasoning
- **AGENT_CONFIG dict**: centralize model choices — easy to swap one model without refactoring
- **Quality heuristics**: simple scoring for code quality, or use LLM-as-judge for production